In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

df = pd.read_csv('bitcoin_data.csv')
df['Returns'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
df = df.dropna()

N = len(df)
confidence_bound = 2 / np.sqrt(N)
print(f"Sample size: {N}")
print(f"95% confidence bound: +/- {confidence_bound:.4f}")

## Part 1: MA(2) Process

**Model:** $X_t = \mu + \varepsilon_t + a_1\varepsilon_{t-1} + a_2\varepsilon_{t-2}$

**Stationarity:** All finite-order MA processes are inherently stationary. The mean and variance do not depend on $t$.

**Invertibility conditions:**
$$|a_2| < 1$$
$$a_2 + a_1 < 1$$
$$a_2 - a_1 < 1$$

**Moments:**
- Mean: $E[X_t] = \mu$
- Variance: $\gamma_0 = \sigma^2(1 + a_1^2 + a_2^2)$
- Autocovariance: $\gamma_1 = \sigma^2(a_1 + a_1 a_2)$, $\gamma_2 = \sigma^2 a_2$, $\gamma_k = 0$ for $k > 2$
- ACF: $\rho_1 = (a_1 + a_1 a_2)/(1 + a_1^2 + a_2^2)$, $\rho_2 = a_2/(1 + a_1^2 + a_2^2)$, $\rho_k = 0$ for $k > 2$

**Key diagnostic:** An MA(q) process has ACF that cuts off sharply to zero after lag $q$.

In [ ]:
# MA(q) suitability: check ACF for a sharp cutoff

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(df['Returns'], lags=20, ax=ax)
ax.set_title('ACF of Bitcoin Log Returns')
ax.set_xlabel('Lags')
ax.set_ylabel('Autocorrelation')
ax.axhline(confidence_bound, color='red', linestyle='--', alpha=0.5, label=f'+/- {confidence_bound:.3f} (95% CI)')
ax.axhline(-confidence_bound, color='red', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

acf_vals = [df['Returns'].autocorr(lag=k) for k in range(1, 6)]
print("ACF values at lags 1-5:")
for i, v in enumerate(acf_vals, 1):
    sig = "*significant*" if abs(v) > confidence_bound else "within noise"
    print(f"  lag {i}: {v:.4f}  ({sig})")
print(f"\nConclusion: No significant spikes, no clear cutoff. MA(q) is not suitable for Bitcoin returns.")

## Part 2: AR(2) Process

**Model:** $X_t = c + b_1 X_{t-1} + b_2 X_{t-2} + \varepsilon_t$

**Invertibility:** All finite-order AR processes are invertible by definition.

**Stationarity conditions:**
$$b_1 + b_2 < 1$$
$$b_2 - b_1 < 1$$
$$|b_2| < 1$$

**Moments:**
- Mean: $\mu = c/(1 - b_1 - b_2)$
- Variance: $\gamma_0 = (1-b_2)\sigma^2 / [(1+b_2)((1-b_2)^2 - b_1^2)]$
- ACF: $\rho_1 = b_1/(1-b_2)$, $\rho_2 = b_1^2/(1-b_2) + b_2$, $\rho_k = b_1\rho_{k-1} + b_2\rho_{k-2}$ for $k \geq 2$

**Key diagnostic:** An AR(p) process has PACF that cuts off sharply to zero after lag $p$, while the ACF decays geometrically.

In [ ]:
# AR(p) suitability: check PACF for a sharp cutoff

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['Returns'], lags=20, ax=axes[0])
axes[0].set_title('ACF of Bitcoin Log Returns')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('Autocorrelation')

plot_pacf(df['Returns'], lags=20, ax=axes[1])
axes[1].set_title('PACF of Bitcoin Log Returns')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('Partial Autocorrelation')

for ax in axes:
    ax.axhline(confidence_bound, color='red', linestyle='--', alpha=0.5)
    ax.axhline(-confidence_bound, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

from statsmodels.tsa.stattools import pacf
pacf_vals = pacf(df['Returns'], nlags=5)[1:]
print("PACF values at lags 1-5:")
for i, v in enumerate(pacf_vals, 1):
    sig = "*significant*" if abs(v) > confidence_bound else "within noise"
    print(f"  lag {i}: {v:.4f}  ({sig})")
print(f"\nConclusion: No significant spikes, no geometric decay in ACF. AR(p) is not suitable for Bitcoin returns.")
print(f"\nBoth results point to the same conclusion: Bitcoin log returns behave like white noise")
print(f"in terms of linear structure. Volatility modeling (GARCH) is needed to capture the remaining dependence.")